# GeFL Class-Balanced — CIFAR-10-LT decision-gate sweep (Kaggle 2×T4, headless)

**Execution mode:** Save Version → **Save & Run All (Commit)**. Kaggle runs this headlessly for up to 12h; close your browser, come back when it's done. Results zip in the Output tab.

**Purpose:** 6-variant sweep on CIFAR-10-LT at IF=0.01, α=0.3, seed 0, 50+50 rounds — enough to decide if the proposal mechanisms improve tail accuracy at all:

| variant             | mech_a | mech_b | support_floor |
|---------------------|--------|--------|---------------|
| baseline            | 0      | 0      | —             |
| a_only              | 1      | 0      | 0             |
| b_only              | 0      | 1      | —             |
| proposed            | 1      | 1      | 0             |
| a_only_floor20      | 1      | 0      | 20            |
| proposed_floor20    | 1      | 1      | 20            |

Two configs run in parallel, one per T4 (via `CUDA_VISIBLE_DEVICES`). 3 pairs × ~4-5h = ~12-15h; the last pair may need a second Save & Run All if the first exhausts the 12h budget.

**Setup (once):**
1. Push code to GitHub (private repo is fine). Fill in `REPO_URL` and `BRANCH` below.
2. Kaggle → New Notebook → Settings → Accelerator = **GPU T4 x2**, Internet = **On**.
3. Paste all cells. Save Version → Save & Run All.
4. Results land in `/kaggle/working/gefl_sweep_results.zip` — download from the notebook's Output tab.

In [ ]:
# ----- Clone the code from GitHub -----
# Fill these in with your own values before Save & Run All.
REPO_URL = 'https://github.com/YOUR_USERNAME/gefl-classbalanced.git'   # ← REPLACE
BRANCH   = 'main'                                                        # ← REPLACE if different
# For a PRIVATE repo, add a fine-grained PAT with 'contents: read' scope
# in Kaggle → Add-ons → Secrets (name it GITHUB_TOKEN), then this cell uses it.

import os, subprocess, sys
from kaggle_secrets import UserSecretsClient

PROJECT_ROOT = '/kaggle/working/project'
if os.path.exists(PROJECT_ROOT):
    subprocess.check_call(['rm', '-rf', PROJECT_ROOT])

clone_url = REPO_URL
try:
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    if token and REPO_URL.startswith('https://github.com/'):
        clone_url = REPO_URL.replace('https://', f'https://x-access-token:{token}@')
    print('Using PAT from Kaggle Secrets (private repo path).')
except Exception:
    print('No GITHUB_TOKEN secret set — assuming public repo.')

subprocess.check_call(['git', 'clone', '--branch', BRANCH, '--depth', '1', clone_url, PROJECT_ROOT])
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

commit = subprocess.check_output(['git', '-C', PROJECT_ROOT, 'rev-parse', 'HEAD'], text=True).strip()
print(f'\nCloned {BRANCH} @ {commit[:12]}')
print('Top-level files:', sorted(os.listdir(PROJECT_ROOT))[:20])

In [ ]:
# ----- Install requirements (Kaggle already has torch/torchvision) -----
req = os.path.join(PROJECT_ROOT, 'requirements.txt')
if os.path.exists(req):
    # Skip torch/torchvision — Kaggle ships GPU-matched builds; forcing pip upgrade breaks CUDA.
    filtered = '/kaggle/working/requirements_no_torch.txt'
    with open(req) as f, open(filtered, 'w') as g:
        for line in f:
            if not line.strip().lower().startswith(('torch', 'torchvision')):
                g.write(line)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', filtered])
print('Deps installed.')

In [ ]:
# ----- GPU sanity check: expect 2 T4s -----
import torch
n = torch.cuda.device_count()
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| devices:', n)
for i in range(n):
    print(f'  cuda:{i} =', torch.cuda.get_device_name(i))
assert n >= 2, 'Enable Settings → Accelerator → GPU T4 x2 (need 2 GPUs for the parallel sweep).'

In [ ]:
# ----- Parallel 2-GPU sweep runner -----
# Six configs total. Pair them so each pair uses both GPUs concurrently.
# 3 pairs × ~4-5h = 12-15h. If pair 3 doesn't finish in one Save & Run All,
# rerun the notebook — completed CSVs are preserved and reused.

import os, sys, subprocess, time, shlex, pandas as pd

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
os.makedirs('./logs', exist_ok=True)

COMMON = [
    '--config', 'configs/cifar10_lt.yaml',
    '--imbalance_factors', '0.01',
    '--dir_params', '0.3',
    '--seeds', '0',
    '--gen_wu_epochs', '50',
    '--epochs', '50',
    '--sample_test', '5',
    '--eval_centralized_upper_bound', '0',
]

def _config(mech, out_csv, extra=None):
    return [sys.executable, 'scripts/sweep.py',
            '--mechanisms', mech,
            '--out_csv', out_csv] + COMMON + (extra or [])

pairs = [
    ('baseline_vs_a', [
        ('./logs/gate_baseline.csv',  _config('baseline', './logs/gate_baseline.csv')),
        ('./logs/gate_a_only.csv',    _config('a_only',   './logs/gate_a_only.csv')),
    ]),
    ('b_vs_proposed', [
        ('./logs/gate_b_only.csv',    _config('b_only',   './logs/gate_b_only.csv')),
        ('./logs/gate_proposed.csv',  _config('proposed', './logs/gate_proposed.csv')),
    ]),
    ('floor_variants', [
        ('./logs/gate_a_floor.csv',   _config('a_only',   './logs/gate_a_floor.csv',   ['--mech_a_support_floor', '20'])),
        ('./logs/gate_prop_floor.csv',_config('proposed', './logs/gate_prop_floor.csv',['--mech_a_support_floor', '20'])),
    ]),
]

def _peek(csv_path):
    """Compact one-line summary of a completed run's final row."""
    try:
        d = pd.read_csv(csv_path).iloc[-1]
        return (f'  {os.path.basename(csv_path):26s} → overall={d.get("acc_overall",0):.3f} '
                f'head={d.get("acc_head",0):.3f} med={d.get("acc_medium",0):.3f} '
                f'tail={d.get("acc_tail",0):.3f} genLA={d.get("gen_label_accuracy",0):.3f}')
    except Exception as e:
        return f'  {csv_path}: (unreadable: {e})'

def _run_pair(pair_name, entries):
    logs = []
    procs = []
    for gpu, (csv_path, cmd) in enumerate(entries):
        if os.path.exists(csv_path):
            print(f'  skip cuda:{gpu} — {csv_path} exists from a prior run')
            procs.append(None); logs.append(None)
            continue
        env = os.environ.copy(); env['CUDA_VISIBLE_DEVICES'] = str(gpu)
        log_path = f'/kaggle/working/{pair_name}_gpu{gpu}.log'
        lf = open(log_path, 'a')
        lf.write(f'\n\n==== {" ".join(map(shlex.quote, cmd))} ====\n'); lf.flush()
        p = subprocess.Popen(cmd, env=env, stdout=lf, stderr=subprocess.STDOUT)
        procs.append(p); logs.append(lf)
        print(f'  cuda:{gpu} → {os.path.basename(csv_path)} (pid={p.pid}, log={log_path})')

    if not any(p for p in procs):
        return

    while any(p is not None and p.poll() is None for p in procs):
        time.sleep(600)   # 10-min heartbeat
        state = ', '.join(
            f'gpu{i}:{"done" if (p is None or p.poll() is not None) else "running"}'
            for i, p in enumerate(procs)
        )
        print(f'    {time.strftime("%H:%M:%S")}  {state}')

    for lf in logs:
        if lf is not None: lf.close()
    for i, p in enumerate(procs):
        if p is not None:
            assert p.returncode == 0, f'gpu{i} failed with code {p.returncode} — see log'

for i, (name, entries) in enumerate(pairs):
    print(f'\n=== Pair {i+1}/{len(pairs)}: {name} ===')
    t0 = time.time()
    _run_pair(name, entries)
    dt = (time.time() - t0) / 60
    print(f'  pair wall: {dt:.1f} min')
    # Peek at whatever CSVs now exist so partial-run notebooks still surface numbers.
    for csv_path, _ in entries:
        print(_peek(csv_path))

In [ ]:
# ----- Aggregate 6 per-run CSVs into one table + run the gate check -----
import pandas as pd, glob

# Map CSV path → variant label (bakes the floor into the name).
label_map = {
    './logs/gate_baseline.csv':    'baseline',
    './logs/gate_a_only.csv':      'a_only',
    './logs/gate_b_only.csv':      'b_only',
    './logs/gate_proposed.csv':    'proposed',
    './logs/gate_a_floor.csv':     'a_only_floor20',
    './logs/gate_prop_floor.csv':  'proposed_floor20',
}

frames = []
for path, label in label_map.items():
    try:
        d = pd.read_csv(path)
        d['variant'] = label
        frames.append(d)
    except FileNotFoundError:
        print(f'(missing: {path})')

df = pd.concat(frames, ignore_index=True)
cols = ['variant', 'seed', 'acc_overall', 'acc_head', 'acc_medium', 'acc_tail', 'gen_label_accuracy']
print(df[cols].sort_values('variant').to_string(index=False))

summary = df.groupby('variant')[['acc_overall', 'acc_head', 'acc_medium', 'acc_tail',
                                  'gen_label_accuracy']].mean()
print('\n=== Per-variant mean (seed 0 only) ===')
print(summary.round(4).to_string())

# Decision-gate check
print('\n=== DECISION GATE ===')
print('Pass criteria: Δtail ≥ +0.03, Δoverall ≥ -0.01, gen_label_acc ≥ 95% of baseline')
try:
    base = summary.loc['baseline']
    for v in summary.index:
        if v == 'baseline':
            print(f'  {v:22s}  (reference)   tail={base["acc_tail"]:.3f}  overall={base["acc_overall"]:.3f}  genLA={base["gen_label_accuracy"]:.3f}')
            continue
        row = summary.loc[v]
        d_tail = row['acc_tail'] - base['acc_tail']
        d_over = row['acc_overall'] - base['acc_overall']
        gen_ok = row['gen_label_accuracy'] >= 0.95 * base['gen_label_accuracy']
        passes = d_tail >= 0.03 and d_over >= -0.01 and gen_ok
        flag = '✓ PASS' if passes else '✗ fail'
        print(f'  {v:22s}  Δtail={d_tail:+.3f}  Δoverall={d_over:+.3f}  gen_ok={str(gen_ok):5s}  {flag}')
except KeyError as e:
    print('missing key:', e)

df.to_csv('./logs/sweep_kaggle_gate_all.csv', index=False)
print('\nMerged CSV: ./logs/sweep_kaggle_gate_all.csv')

In [ ]:
# ----- Bundle everything for download -----
import shutil

bundle_dir = '/kaggle/working/gefl_sweep_bundle'
if os.path.exists(bundle_dir):
    shutil.rmtree(bundle_dir)
os.makedirs(bundle_dir)

for src_sub in ('logs',):
    src_path = os.path.join(PROJECT_ROOT, src_sub)
    if os.path.exists(src_path):
        shutil.copytree(src_path, os.path.join(bundle_dir, src_sub))

shutil.make_archive('/kaggle/working/gefl_sweep_results', 'zip', bundle_dir)
print('Bundle at /kaggle/working/gefl_sweep_results.zip — download from the notebook sidebar.')